## A notebook to analyze groups of croparrays  

### Read in, combine, and making measurements 

In [1]:
import croparray as ca
from pathlib import Path
%gui qt

In [2]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

# Files & Directories
DATA_DIR_1 = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/NoZNF598/Stacks/CropArrays")
DATA_DIR_2 = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays")
LABELS = ['-ZNF598','+ZNF598']
OUTPUT = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/HT-analysis") 

nc_files_1 = sorted(DATA_DIR_1.glob("*.nc"))
nc_files_2 = sorted(DATA_DIR_2.glob("*.nc"))

# Measurement parameters
REF_CH = 0  # Optional reference channel for best_z_proj (if use_zc=False, i.e. when Z not tracked)
DISK_R = 5  # Radius of disk in pixels to measure signal
DISK_BG = 7 # Radius of single pixel ring around disk to measure background 
ROLL_N = 1  # Rolling z-average for signal measurement.

In [3]:
my_ca = ca.build.open_measure_concat(
    groups=[nc_files_1, nc_files_2],
    dims=["exp", "fov"],
    labels=[["-ZNF598", "+ZNF598"], None],
    measure_kwargs=dict(ref_ch=REF_CH, disk_r=DISK_R, disk_bg=DISK_BG, roll_n=ROLL_N,drop_int=True, # drop full z-stack to save memory, keeping best_z_proj
    ),
    open_as = "croparray",  # can be croparray or trackarray 
    join="outer",
)

[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.
[CropArray] No sidecars found.


In [5]:
# Create binary masks from "best_z" so can measure morphological properties. 
my_ca.ops.apply(
    ca.tools.binarize_crop_manual,
    channels=[0],               # Channel used to generate mask
    source="best_z",            # Image layer to threshold (typically z-projected signal)
    out_name="ch{ch}_mask",     # Output mask variable name (→ "ch0_mask")

    # Recommended parameters forwarded directly to binarize_crop_manual(...)
    func_kwargs=dict(
        q=0.45,                 # Threshold at 45% after intensity normalization
        q_range=(0.02, 0.999),  # Normalize intensities using 2%–99.9% quantile range
        q_positive_only=True,   # Ignore negative pixels (important for best_z)
        close_px=1,             # Morphological closing (bridge small gaps)
        smooth_px=0,            # Light smoothing of mask edges
        fill_holes=False,       # Do not fill internal holes
        return_uint8=True,      # Store mask as uint8 (0/1)
        morph_px=0,             # Apply a dilation of final mask (morph_pix > 0; <0 for erosion)
    ),
);

In [6]:
# Check masks with napari
viewer, layers = my_ca.napari.montage_viewer(
    row="n",
    col="t",
    show=("best_z", "ch0_mask"),
    #ch=[0, 1, 2],  # RGB
    colormaps={"ch0_mask": "magenta","best_z": "green"},
    image_contrast=[0.5,99.5],
    show_tile_text=False,
)

In [6]:
# Make measurements of desired properties 
# See https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops
my_ca.measure.mask_props(source ="ch0_mask", props=['eccentricity','major_axis_length_px','minor_axis_length_px']);

In [7]:
my_ca.ds.attrs["concat_meta_json"]

'{"base_name": "PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome", "dims": ["exp", "fov"], "n_files": 14}'

In [8]:
# =======================================
# SAVE YOUR WORK TO A NEW CROP/TRACKARRAY
# =======================================

my_file = ca.io.save_croparray(my_ca, output_dir=OUTPUT, ext='.nc')
my_file

C:\Users\tstasevi\Documents\GitHub\croparray\croparray\io.py:467: SerializationWarning: saving variable xc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
c:\Users\tstasevi\AppData\Local\anaconda3\envs\croparray_env\lib\site-packages\xarray\core\duck_array_ops.py:253: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)
C:\Users\tstasevi\Documents\GitHub\croparray\croparray\io.py:467: SerializationWarning: saving variable yc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
C:\Users\tstasevi\Documents\GitHub\croparray\croparray\io.py:467: SerializationWarning: saving variable zc_pix with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_to_save.to_netcdf(str(out_path), **kwargs)
C:\Users\tstasevi\Documents\GitHub\croparray\croparray\io.py:467: Ser

WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/HT-analysis/PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov.nc')

In [9]:
# check your saved data; note concatenation history is saved
temp = ca.tools.open_croparray(my_file)
temp.ds

[CropArray] Found 1 sidecar(s).
[CropArray] Checking sidecar: C:\Users\tstasevi\Documents\Tim-Marianas\TrnlWithZNF598\20260210\HT-analysis\PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov__manual__bad.nc
[CropArray] Merging sidecar C:\Users\tstasevi\Documents\Tim-Marianas\TrnlWithZNF598\20260210\HT-analysis\PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov__manual__bad.nc with vars: ['bad']
[CropArray] Merging sidecars into CropArray dataset.


<xarray.Dataset> Size: 1GB
Dimensions:                         (n: 278, t: 46, y: 21, x: 21, fov: 14,
                                     exp: 2, ch: 1, z: 1)
Coordinates:
  * n                               (n) int16 556B 0 1 2 3 4 ... 274 275 276 277
  * t                               (t) int32 184B 0 1 2 3 4 ... 41 42 43 44 45
  * y                               (y) float64 168B -0.74 -0.666 ... 0.666 0.74
  * x                               (x) float64 168B -0.74 -0.666 ... 0.666 0.74
  * fov                             (fov) object 112B 'PB_BG_Max_ALL_Cell_HT ...
  * exp                             (exp) object 16B '-ZNF598' '+ZNF598'
  * z                               (z) float64 8B 0.0
  * ch                              (ch) int32 4B 0
Data variables: (12/25)
    ch0_mask                        (exp, fov, n, t, x, y) int8 158MB ...
    xc                              (exp, fov, n, t, ch) float32 1MB ...
    yc                              (exp, fov, n, t, ch) float32 1MB ...
    zc                              (exp, fov, n, t, ch) float32 1MB ...
    xc_pix                          (exp, fov, n, t, ch) int16 716kB ...
    yc_pix                          (exp, fov, n, t, ch) int16 716kB ...
    ...                              ...
    ch0_mask__eccentricity          (exp, fov, n, t) float64 3MB ...
    ch0_mask__major_axis_length_px  (exp, fov, n, t) float64 3MB ...
    ch0_mask__minor_axis_length_px  (exp, fov, n, t) float64 3MB ...
    best_z                          (exp, fov, n, ch, t, y, x) float64 1GB ...
    signal                          (exp, fov, n, ch, t) float64 3MB ...
    bad                             (fov, exp, n, t) int8 358kB 0 0 0 ... 0 0 0
Attributes: (12/16)
    name:                      PB_BG_Max_ALL_HI_at_5 - Position 1_XY177077284...
    date:                      2026-02-10
    xy_pad:                    10
    z_pad:                     1
    dx:                        0.074
    dy:                        0.074
    ...                        ...
    signal_units:              a.u.
    croparray_schema_version:  2.0
    notes:                     A crop array with ch0 = 12xSunTag-KDM5B-XBP1(S...
    provenance_json:           {\n  "timestamp": "2026-03-06T17:33:44",\n  "d...
    concat_meta_json:          {"base_name": "PB_BG_Max_ALL_HI_at_5_-_Positio...
    filename:                  PB_BG_Max_ALL_HI_at_5_-_Position_1_XY177077284...

### Manual filtering of saved croparray

In [1]:
import croparray as ca
from pathlib import Path
%gui qt

In [2]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

INPUT_FILE = Path('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/HT-analysis/PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov.nc')

In [3]:
# Need to open this way to get proper doc strings:
from croparray.trackarray.object import CropArray
my_ca: CropArray = ca.io.open_croparray(INPUT_FILE, as_object=True)

[CropArray] Found 1 sidecar(s).
[CropArray] Checking sidecar: C:\Users\tstasevi\Documents\Tim-Marianas\TrnlWithZNF598\20260210\HT-analysis\PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov__manual__bad.nc
[CropArray] Merging sidecar C:\Users\tstasevi\Documents\Tim-Marianas\TrnlWithZNF598\20260210\HT-analysis\PB_BG_Max_ALL_HI_at_5_-_Position_1_XY1770772846.ome__14exp_fov__manual__bad.nc with vars: ['bad']
[CropArray] Merging sidecars into CropArray dataset.


In [4]:
# Quick and user-friendly manual filtering of tracks with napari 
# Create and save filter names (e.g. a 'bad' filter to remove noisy crops/tracks)
# Can also create filters like 'toi' to highlight tracks of interest  
filter_name = "bad"
viewer, layers, ft = my_ca.napari.manual_filter_montage(
    row="n",  # can be 'track_id' for trackarrays or 'n' for croparrays               
    col="t",  
    filter_name=filter_name,
    show=("best_z", "ch0_mask"),
    ch=0,
    colormaps={"ch0_mask":"magenta", "best_z":"green", filter_name:"yellow"},
    output_dir= INPUT_FILE.parent,
    show_tile_text=False, # adds layer w/ crop coords; takes memory,
    show_click_info=True, # useful for debugging
)

In [ ]:
# Test your filter:
my_ta.where(my_ta.ds[filter_name] == 0).napari.montage_viewer(row="track_id", col="t",show=("best_z","ch0_mask"),ch=0,show_tile_text=False)

(Viewer(camera=Camera(center=(0.0, np.float64(6982.0), np.float64(104.5)), zoom=np.float64(0.05442176870748299), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(0.0), np.float64(3.0), 0.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=4, ndisplay=2, order=(0, 1, 2, 3), axis_labels=('0', '1', '2', '3'), rollable=(True, True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(1.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(6.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(13964.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(209.0), step=np.float64(1.0))), margin_left=(0.0, 0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0, 0.0), poi